# Báo cáo tài chính — BCTC PDF (Semi-structured Data)

Notebook điều khiển riêng **luồng BCTC**: discovery link PDF trong tin VCI → tải → manifest Parquet.

| Đầu vào | Mô tả |
|----------|----------|
| `listing.parquet` | Danh sách mã + sàn từ pipeline có cấu trúc |
| `.env` | `VNSTOCK_API_KEY` |

| Đầu ra | Đường dẫn |
|---------|----------|
| PDF files | `data-lake/raw/Semi_Structure_Data/bctc/pdf/exchange=…/symbol=…/year=…/q=…/*.pdf` |
| Manifest (ngày) | `data-lake/raw/Semi_Structure_Data/bctc/manifest/date=<ngày>/PART-000.parquet` |
| Master manifest | `data-lake/raw/Semi_Structure_Data/bctc/master/bctc_files.parquet` |

In [ ]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import configure_logging, load_dotenv_from_project_root
from ingestion.semi_structure_data import BctcPdfConfig, ingest_bctc_pdfs

configure_logging()
load_dotenv_from_project_root()
print("[OK] setup")

## Kiểm tra listing (input bắt buộc)

In [ ]:
import pandas as pd

cfg = BctcPdfConfig()
listing_path = cfg.resolved_listing_parquet()
print(f"listing path: {listing_path}")
print(f"exists: {listing_path.is_file()}")

if listing_path.is_file():
    listing_df = pd.read_parquet(listing_path)
    print(f"total symbols: {listing_df['symbol'].nunique()}")
    if "exchange" in listing_df.columns:
        print(listing_df["exchange"].value_counts().to_dict())
else:
    print("Chưa có listing.parquet — chạy ingest_structure_data_manager.ipynb trước!")

## Cấu hình

In [ ]:
bctc_cfg = BctcPdfConfig(
    quarters_back=8,
    exchanges=["HSX", "HNX", "UPCOM"],
    discovery_max_news_pages=15,
    max_symbols_per_run=None,       # None = tất cả mã trong listing
    rate_limit_rpm=20,
)

print(f"run_date          : {bctc_cfg.run_date}")
print(f"pdf_root          : {bctc_cfg.pdf_root}")
print(f"master_manifest   : {bctc_cfg.master_manifest_path}")
print(f"exchanges         : {bctc_cfg.exchanges}")
print(f"quarters_back     : {bctc_cfg.quarters_back}")
print(f"max_symbols_per_run: {bctc_cfg.max_symbols_per_run or 'ALL'}")
print(f"discovery_pages   : {bctc_cfg.discovery_max_news_pages}")

## Chạy ingest BCTC

In [ ]:
result = ingest_bctc_pdfs(
    bctc_cfg,
    refresh_listing=False,
)
result

## Kiểm tra kết quả

In [ ]:
import pandas as pd

master = bctc_cfg.master_manifest_path
if master.is_file():
    mdf = pd.read_parquet(master)
    ok = mdf[mdf["error"].isna() | (mdf["error"] == "")]
    print(f"Master manifest: {len(mdf)} rows ({len(ok)} downloaded OK)")
    if "symbol" in mdf.columns:
        print(f"Symbols covered: {mdf['symbol'].nunique()}")
    if "exchange" in mdf.columns:
        print(f"By exchange: {mdf['exchange'].value_counts().to_dict()}")
    display(mdf[["symbol","exchange","year","quarter","title","http_status","error"]].tail(10))
else:
    print("Chưa có master manifest.")

manifest_part = result.get("manifest_partition", "")
if manifest_part:
    pdf = pd.read_parquet(manifest_part)
    print(f"\nPartition hôm nay: {len(pdf)} rows")
else:
    print("Không có manifest mới hôm nay.")

## Danh sách PDF đã tải (trên đĩa)

In [ ]:
from pathlib import Path

pdfs = sorted(bctc_cfg.pdf_root.rglob("*.pdf"))
print(f"Tổng PDF trên đĩa: {len(pdfs)}")
for p in pdfs[:20]:
    rel = p.relative_to(bctc_cfg.pdf_root)
    size_kb = p.stat().st_size / 1024
    print(f"  {rel}  ({size_kb:.0f} KB)")
if len(pdfs) > 20:
    print(f"  ... và {len(pdfs) - 20} file khác")